# Agentic RAG vs Naive RAG

**任务**：基于一个 *关于 LLM Agent 的迷你中文知识库* 回答问题。

我们对比两种实现：

- **Naive RAG**：一次 embedding 检索 → top-k → 拼 prompt → 答。
- **Agentic RAG**：把 `search_docs(query, k)` 注册为工具，Claude 自主决定调用次数、可以改写 query。

数据集是手工构造的 *需要多跳 / 需要改写 query* 的题，用以暴露 Naive RAG 的局限。

In [ ]:
import os, sys, json, numpy as np
sys.path.append(os.path.abspath('../..'))
from utils.llm_client import LLMClient
from anthropic import Anthropic
client = LLMClient()
anthropic = Anthropic()
MODEL = client.model

## 1. 迷你知识库 + Embedding

In [ ]:
DOCS = [
    'ReAct（Yao 2022）让 LLM 交替输出 Thought/Action/Observation，把推理与工具调用编织在一起。',
    'Reflexion（Shinn 2023）通过自然语言反思在多 episode 间积累经验，是免梯度的 verbal RL。',
    'Tree of Thoughts（Yao 2023）把 CoT 的线性思路扩展为搜索树，由 LLM 当估值函数。',
    'LATS（Zhou 2024）= ToT + ReAct + Reflexion，用 MCTS 串起搜索、行动、反思。',
    'Generative Agents（Park 2023）用观察-反思-计划三层记忆机制让 25 个 LLM 角色在小镇里涌现社会行为。',
    'MemGPT（Packer 2023）借 OS 虚拟内存的思想，让 LLM 通过工具自管理 working / archival memory。',
    'Self-RAG（Asai 2024）让 LLM 输出 reflection token 决定是否检索、引用是否充分。',
    'Corrective RAG（CRAG, Yan 2024）在检索后插入 evaluator，结果不够时转 web 搜索兜底。',
    'Anthropic 在 2024 提出 MCP（Model Context Protocol），把 LLM ↔ 工具/数据接口标准化。',
    'CodeAct（Wang 2024）把 agent 的 action 表达统一为可执行 Python 代码，比 JSON tool_use 表达力更强。',
    'AutoGen（Microsoft 2023）以多 agent 对话为核心抽象，支持灵活的 conversational programming。',
    'MetaGPT（Hong 2023）让多个角色 agent 模拟软件公司 SOP，把 SDLC 拆给不同 agent。',
    'GAIA（Mialon 2023）是 Meta 提出的真实世界通用 agent benchmark，强调多步推理与工具使用。',
    'SWE-bench（Jimenez 2023）从真实 GitHub issue 提取测试用例，评估 agent 修复 bug 能力。',
    'WebArena（Zhou 2023）提供可重现的 web 任务环境，覆盖电商/论坛/CMS 等真实站点克隆。',
    'GRPO（DeepSeek 2024）相比 PPO 不需要 value model，用组内归一化 advantage，省内存。',
    'DAPO（ByteDance 2025）在 GRPO 基础上做了 clip-higher 等改进，AIME 2024 达 50 分。',
    'SkyRL-Agent（2025）针对多轮长程 agent 任务给出异步 dispatch + 工具集成的高效 RL 训练框架。',
    'MapAgent（2025）是分层多 agent 框架，由高层 planner 拆任务，专门 map-tool agent 并行调地图 API。',
    'PReP（2024）用 perceive-reflect-plan 三阶段让 fine-tuned LLaVA 在城市导航中无指令也能找到路。',
]
len(DOCS)

In [ ]:
# 用 sentence-transformers 离线 embedding；如果不希望下载模型，
# 可以替换为 anthropic 的 embedding API（暂未提供）或简单 TF-IDF baseline。
from sentence_transformers import SentenceTransformer
_st = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
DOC_EMB = _st.encode(DOCS, normalize_embeddings=True)
DOC_EMB.shape

In [ ]:
def search_docs(query: str, k: int = 3):
    q = _st.encode([query], normalize_embeddings=True)[0]
    sims = DOC_EMB @ q
    idx = np.argsort(-sims)[:k]
    return [{'rank': i+1, 'score': float(sims[j]), 'text': DOCS[j]} for i, j in enumerate(idx)]

search_docs('ToT 是什么', k=3)

## 2. Naive RAG

In [ ]:
def naive_rag(question: str, k: int = 3) -> str:
    hits = search_docs(question, k=k)
    ctx = '\n'.join(f'[{h["rank"]}] {h["text"]}' for h in hits)
    prompt = (
        '你是一名 LLM Agent 助教。请仅根据下面的检索段落回答问题，'
        '若证据不足请明确说明。\n\n检索段落：\n' + ctx +
        f'\n\n问题：{question}\n请简洁回答（≤60 字）：'
    )
    return client.chat([{'role': 'user', 'content': prompt}])['text']

print(naive_rag('LATS 跟 ToT 是什么关系？'))

## 3. Agentic RAG（把 search_docs 注册成 tool）

In [ ]:
TOOLS = [{
    'name': 'search_docs',
    'description': '在 LLM Agent 中文知识库中检索片段。可多次调用，每次用不同 query 精化。',
    'input_schema': {
        'type': 'object',
        'properties': {
            'query': {'type': 'string'},
            'k': {'type': 'integer', 'default': 3},
        },
        'required': ['query'],
    },
}]

SYSTEM = (
    '你是一名 LLM Agent 助教。可以多次调用 search_docs 来收集证据。'
    '每条回答必须基于检索证据，并在最后用 [n] 标注引用 rank。'
)

def agentic_rag(question: str, max_steps: int = 6, verbose: bool = False):
    messages = [{'role': 'user', 'content': question}]
    for step in range(max_steps):
        resp = anthropic.messages.create(
            model=MODEL, max_tokens=512, tools=TOOLS, system=SYSTEM, messages=messages,
        )
        if verbose:
            for b in resp.content:
                tag = b.type
                if tag == 'tool_use':
                    print(f'[{step}] call {b.name}({b.input})')
                elif tag == 'text':
                    print(f'[{step}] text: {b.text[:80]}...')
        if resp.stop_reason != 'tool_use':
            return ''.join(b.text for b in resp.content if b.type == 'text')
        messages.append({'role': 'assistant', 'content': resp.content})
        results = []
        for b in resp.content:
            if b.type == 'tool_use':
                hits = search_docs(**b.input)
                results.append({'type': 'tool_result', 'tool_use_id': b.id,
                                'content': json.dumps(hits, ensure_ascii=False)})
        messages.append({'role': 'user', 'content': results})
    return '[max_steps]'

print(agentic_rag('LATS 跟 ToT 是什么关系？', verbose=True))

## 4. 难题对比

下面这些问题需要 *多跳* 或 *先查再改写*，Naive RAG 通常会漏。

In [ ]:
QUESTIONS = [
    '把 ToT、ReAct、Reflexion 三者结合的工作叫什么？它的搜索算法是什么？',
    '哪两个工作分别提出了在检索失败时调 web 搜索 / 在生成时插 reflection token？',
    'Generative Agents 的三层记忆机制具体是哪三层？这种思想被哪个 OS 类比框架借鉴了？',
    '哪个 RL 算法不需要 value model？哪个是它的改进版？',
]
for q in QUESTIONS:
    print('Q:', q)
    print('Naive:', naive_rag(q))
    print('Agent:', agentic_rag(q))
    print('---')

## 5. 思考

- Naive RAG 在「单跳、措辞与文档相近」的题目上表现尚可，但遇到 *多跳 / 改写* 就掉链子。
- Agentic RAG 通过多次检索 + 自主改写，能解出更多问题，但 token 开销 ≈ 2-4×。
- 实际系统建议：用 *router* 决定哪些 query 走 naive，哪些升级到 agentic（如基于 query 长度 / 子句数 / 不确定度）。

## 进阶练习

1. 加 BM25 + 向量混合检索（rank-bm25 包），对比命中率提升。
2. 加 reflection 循环：让 agent 在每次答完后自检 `IsSup`，证据不足就再查一轮。
3. 实现 Router-RAG：先让 LLM 判断 query 类型，再决定调多少次检索。